# Prerequisites

## Load Bronze data to dataframe

In [0]:
%sql
use catalog ecommerce;
create schema if not exists silver;


In [0]:
customers_df = spark.read.table("bronze.customers")
customers_df.limit(5).display()

In [0]:
customers_df.printSchema()

# Transformations

## Remove Duplicate

In [0]:
from pyspark.sql.functions import *
customers_df.groupBy('customer_id').count().filter(col("count")>1).display()

In [0]:
customers_df = customers_df.dropDuplicates()

## Keep the latest record where appropriate.

In [0]:
from pyspark.sql.window import Window
window_spec = (Window.partitionBy('customer_id').orderBy(col('updated_at').desc()))
customers_df = customers_df.withColumn('rnk', row_number().over(window_spec)).filter(col('rnk')==1).drop('rnk')
customers_df.display()

## Trim whitespace & Convert email to lowercase.

In [0]:
customers_df = customers_df.withColumn("email", lower(trim(col("email"))))

In [0]:
customers_df.limit(5).display()

## Validate email format

In [0]:
email_regex = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
customers_df = customers_df.withColumn(
    "is_valid_email", col("email").rlike(email_regex)
).filter(col("is_valid_email") == True)
customers_df.limit(5).display()

## Handle missing email/phone.

In [0]:
customers_df = (
    customers_df.withColumn(
        "missing_email", col("email").isNull() | (trim(col("email")) == "")
    )
    .withColumn("missing_phone", col("phone").isNull() | (trim(col("phone")) == ""))
    .filter(col("missing_email") == False)
    .filter(col("missing_phone") == False)
    .filter(col("customer_id").isNotNull())
)
customers_df.limit(5).display()

## Standardize city/state

In [0]:
customers_df = (
    customers_df.withColumn("city", initcap(trim(col("city"))))
    .withColumn("state", initcap(trim(col("state"))))
    .withColumn("country", initcap(trim(col("country"))))
)
customers_df.limit(5).display()


## Validate customer status

In [0]:
customers_df = customers_df.filter(col("customer_status").isin ("ACTIVE","INACTIVE"))
customers_df.limit(5).display()

# Create Silver table

## Write to silver schema

In [0]:
customers_df.write.format("delta").mode("overwrite").saveAsTable("ecommerce.silver.customers")

## Read silver.customers table

In [0]:
spark.read.table("ecommerce.silver.customers").display()